# SPROUT Training Notebook
## Symptom-centric Prototypical Representation Optimization and Uncertainty-aware Tuning

Run each cell one by one in Anaconda Navigator's Jupyter Notebook.

## Cell 1: Install Required Packages

In [10]:
# Run this cell first to install packages (uncomment if needed)
!pip install torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow tqdm timm

  Using cached timm-1.0.29-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-1.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached timm-1.0.29-py3-none-any.whl (2.6 MB)
   ---------------------------------------- 0.0/798.3 kB ? eta -:--:--
   --------------------------------------- 798.3/798.3 kB 11.1 MB/s eta 0:00:00
Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl (4.0 MB)

  Attempting uninstall: click

    Found existing installation: click 8.1.8

    Uninstalling click-8.1.8:

      Successfully uninstalled click-8.1.8

   ------------------------ --------------- 3/5 [huggingface_hub]
   ------------------------ --------------- 3/5 [huggingface_hub]
   ------------------------ --------------- 3/5 [huggingface_hub]
   ------------------------ --------------- 3/5

In [8]:
pip install torch

In [5]:
pip install torchvision

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------------------------------ --------- 1.0/1.4 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 6.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Cell 2: Import Libraries

In [11]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.14.0+cpu
CUDA available: False


## Cell 3: Set Random Seeds & Device

In [12]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Cell 4: Define Dataset Class

In [15]:
class LeafDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None, split='train'):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.split_dir = self.root_dir / split
        
        self.classes = sorted([d.name for d in self.split_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.image_paths = []
        self.labels = []
        
        for class_name in self.classes:
            class_dir = self.split_dir / class_name
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG']:
                for img_path in class_dir.glob(f'**/{ext}'):
                    self.image_paths.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])
        
        self.labels = torch.tensor(self.labels)
        self.indices_by_class = {}
        for class_idx in range(len(self.classes)):
            self.indices_by_class[class_idx] = torch.where(self.labels == class_idx)[0].tolist()
        
        print(f"Found {len(self.image_paths)} images across {len(self.classes)} classes in {split} set")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            image = torch.zeros((3, 224, 224))
        return image, label
    
    def get_classes(self):
        return self.classes

## Cell 5: Define Model Components

In [16]:
class EmbeddingNetwork(nn.Module):
    def __init__(self, input_dim, embed_dim=128, hidden_dims=[512, 256], dropout_rate=0.3):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dims[0]))
        layers.append(nn.BatchNorm1d(hidden_dims[0]))
        layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Dropout(p=dropout_rate))
        for i in range(len(hidden_dims) - 1):
            layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            layers.append(nn.BatchNorm1d(hidden_dims[i+1]))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(p=dropout_rate))
        layers.append(nn.Linear(hidden_dims[-1], embed_dim))
        self.embedding_layers = nn.Sequential(*layers)

    def forward(self, x):
        embeddings = self.embedding_layers(x)
        return F.normalize(embeddings, p=2, dim=1)


class PrototypeModule(nn.Module):
    def __init__(self, embed_dim=128, num_refinement_steps=3, num_heads=4):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_refinement_steps = num_refinement_steps
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads, dropout=0.1, batch_first=True
        )
        self.attention = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim // 2),
            nn.LayerNorm(embed_dim // 2),
            nn.GELU(),
            nn.Linear(embed_dim // 2, 1)
        )
        self.refinement_network = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, embed_dim)
        )
        self.attn_temperature = nn.Parameter(torch.ones(1))

    def generate_initial_prototypes(self, support_embeddings, support_labels):
        classes = torch.unique(support_labels)
        prototypes = []
        for c in classes:
            class_mask = (support_labels == c)
            class_embeddings = support_embeddings[class_mask]
            prototype = torch.mean(class_embeddings, dim=0) if len(class_embeddings) > 0 else torch.zeros(support_embeddings.size(1), device=support_embeddings.device)
            prototypes.append(prototype)
        return torch.stack(prototypes)

    def compute_attention_weights(self, support_embeddings, prototype, support_labels, class_idx):
        class_mask = (support_labels == class_idx)
        class_embeddings = support_embeddings[class_mask]
        if len(class_embeddings) == 0:
            return None
        prototype_expanded = prototype.unsqueeze(0).expand(class_embeddings.size(0), -1)
        attention_input = torch.cat([class_embeddings, prototype_expanded], dim=1)
        attention_scores = self.attention(attention_input)
        attention_scores = attention_scores / self.attn_temperature
        attention_weights = F.softmax(attention_scores, dim=0)
        return attention_weights, class_embeddings

    def refine_prototype(self, prototype, support_embeddings, support_labels, class_idx):
        attention_result = self.compute_attention_weights(support_embeddings, prototype, support_labels, class_idx)
        if attention_result is None:
            return prototype
        attention_weights, class_embeddings = attention_result
        weighted_avg = torch.sum(attention_weights * class_embeddings, dim=0)
        class_embeddings_seq = class_embeddings.unsqueeze(0)
        prototype_query = prototype.unsqueeze(0).unsqueeze(0)
        attn_output, _ = self.multihead_attn(
            query=prototype_query, key=class_embeddings_seq, value=class_embeddings_seq
        )
        attn_refined = attn_output.squeeze(0).squeeze(0)
        refinement_input = torch.cat([prototype, weighted_avg + attn_refined], dim=0)
        refinement_delta = self.refinement_network(refinement_input.unsqueeze(0)).squeeze(0)
        refined_prototype = prototype + refinement_delta
        return F.normalize(refined_prototype, p=2, dim=0)

    def forward(self, support_embeddings, support_labels):
        if len(support_embeddings) == 0:
            return torch.zeros((0, self.embed_dim), device=support_embeddings.device)
        prototypes = self.generate_initial_prototypes(support_embeddings, support_labels)
        classes = torch.unique(support_labels)
        for _ in range(self.num_refinement_steps):
            refined_prototypes = []
            for i, c in enumerate(classes):
                if i < len(prototypes):
                    refined_prototypes.append(self.refine_prototype(prototypes[i], support_embeddings, support_labels, c))
            prototypes = torch.stack(refined_prototypes) if refined_prototypes else prototypes
        return prototypes


Model components defined successfully!


## Cell 6: Define SPROUT Model & Loss

In [17]:
class SPROUT(nn.Module):
    def __init__(self, num_classes, backbone="resnet50", embed_dim=128,
                 hidden_dims=[512, 256], num_refinement_steps=3, temperature=10.0,
                 dropout_rate=0.3, num_heads=4):
        super().__init__()
        self.feature_extractor = FeatureExtractor(backbone=backbone)
        feature_dim = self.feature_extractor.get_feature_dim()
        self.embedding_network = EmbeddingNetwork(input_dim=feature_dim, embed_dim=embed_dim, hidden_dims=hidden_dims, dropout_rate=dropout_rate)
        self.prototype_module = PrototypeModule(embed_dim=embed_dim, num_refinement_steps=num_refinement_steps, num_heads=num_heads)
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(temperature)))

    def forward(self, query_images, support_images=None, support_labels=None):
        query_features = self.feature_extractor(query_images)
        query_embeddings = self.embedding_network(query_features)
        if support_images is not None and support_labels is not None:
            support_features = self.feature_extractor(support_images)
            support_embeddings = self.embedding_network(support_features)
            prototypes = self.prototype_module(support_embeddings, support_labels)
            logits = -self.compute_distances(query_embeddings, prototypes)
            return logits, prototypes
        return query_embeddings

    def compute_distances(self, embeddings, prototypes):
        embeddings_expanded = embeddings.unsqueeze(1)
        prototypes_expanded = prototypes.unsqueeze(0)
        distances = torch.sum((embeddings_expanded - prototypes_expanded) ** 2, dim=2)
        temperature = torch.exp(self.log_temperature)
        return distances / temperature

    def extract_embeddings(self, images):
        self.eval()
        with torch.no_grad():
            features = self.feature_extractor(images)
            embeddings = self.embedding_network(features)
        return embeddings


SPROUT model and loss defined successfully!


## Cell 7: Load Dataset

In [18]:
# Path to dataset
DATA_DIR = r"C:\Users\HP\SPROUT-main\data\plantvillage"

# Transforms with stronger augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15))
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = LeafDiseaseDataset(DATA_DIR, transform=train_transform, split="train")
test_dataset = LeafDiseaseDataset(DATA_DIR, transform=test_transform, split="test")
num_classes = len(train_dataset.get_classes())
print("Classes:", train_dataset.get_classes())
print("Number of classes:", num_classes)


Found 18311 images across 10 classes in train set
Found 4585 images across 10 classes in test set

Classes: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot']
Number of classes: 10


## Cell 8: Create Model

In [19]:
# Hyperparameters
N_WAY = 5          # Number of classes per episode
K_SHOT = 5         # Number of support examples per class
N_QUERY = 15       # Number of query examples per class
NUM_EPISODES = 100 # Episodes per epoch
NUM_EPOCHS = 25    # Number of epochs (increased)
LR = 0.001         # Learning rate
WEIGHT_DECAY = 1e-5
WARMUP_EPOCHS = 3

# Create model with improved architecture
num_classes = len(train_dataset.get_classes())
model = SPROUT(
    num_classes=num_classes,
    backbone="resnet50",
    embed_dim=128,
    hidden_dims=[512, 256],
    num_refinement_steps=3,
    temperature=10.0,
    dropout_rate=0.3,
    num_heads=4
)

# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing with warm restarts scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Model created on", device)
print("Total parameters:", sum(p.numel() for p in model.parameters()))


C:\Users\HP\anaconda3\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:09<00:00, 11.1MB/s]


Model created with 24,788,801 parameters
Number of classes: 10


## Cell 9: Training Function

In [20]:
def create_episode(dataset, n_way, k_shot, n_query):
    """Create a single episode with support and query sets"""
    indices_by_class = dataset.indices_by_class
    available_classes = [c for c in indices_by_class if len(indices_by_class[c]) >= k_shot + 1]
    
    if len(available_classes) < n_way:
        available_classes = [c for c in indices_by_class if len(indices_by_class[c]) >= 2]
        n_way = min(n_way, len(available_classes))
    
    selected_classes = random.sample(available_classes, n_way)
    
    support_images, support_labels = [], []
    query_images, query_labels = [], []
    
    for new_label, class_idx in enumerate(selected_classes):
        indices = indices_by_class[class_idx]
        random.shuffle(indices)
        
        support_indices = indices[:k_shot]
        query_indices = indices[k_shot:k_shot + n_query]
        
        for idx in support_indices:
            img, _ = dataset[idx]
            support_images.append(img)
            support_labels.append(new_label)
        
        for idx in query_indices:
            img, _ = dataset[idx]
            query_images.append(img)
            query_labels.append(new_label)
    
    support_images = torch.stack(support_images).to(device)
    support_labels = torch.tensor(support_labels).to(device)
    query_images = torch.stack(query_images).to(device)
    query_labels = torch.tensor(query_labels).to(device)
    
    return support_images, support_labels, query_images, query_labels

print("Episode creation function defined!")

Episode creation function defined!


## Cell 10: Train the Model (Run This!)

In [ ]:
# Create output directory
os.makedirs("./results", exist_ok=True)

# Track metrics
train_accuracies = []
train_losses = []

print("Starting SPROUT Training...")
print("=" * 50)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 30)
    
    episode_accuracies = []
    episode_losses = []
    
    for episode in tqdm(range(NUM_EPISODES), desc="Training"):
        model.train()
        optimizer.zero_grad()
        
        # Create episode
        support_images, support_labels, query_images, query_labels = create_episode(
            train_dataset, N_WAY, K_SHOT, N_QUERY
        )
        
        # Forward pass
        logits, prototypes = model(query_images, support_images, support_labels)
        
        # Get initial prototypes for loss
        with torch.no_grad():
            support_features = model.feature_extractor(support_images)
            support_embeddings = model.embedding_network(support_features)
            initial_prototypes = model.prototype_module.generate_initial_prototypes(support_embeddings, support_labels)
        
        # Compute loss
        loss, loss_components = criterion(logits, query_labels, prototypes, initial_prototypes, support_embeddings, support_labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        _, predicted = torch.max(logits.data, 1)
        accuracy = (predicted == query_labels).float().mean().item()
        
        episode_accuracies.append(accuracy)
        episode_losses.append(loss_components['total_loss'])
    
    # Update learning rate
    scheduler.step()
    
    # Epoch summary
    epoch_acc = np.mean(episode_accuracies)
    epoch_loss = np.mean(episode_losses)
    train_accuracies.append(epoch_acc)
    train_losses.append(epoch_loss)
    
    print(f"Epoch {epoch+1} - Accuracy: {epoch_acc:.4f}, Loss: {epoch_loss:.4f}")
    
    # Save checkpoint
    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        torch.save(model.state_dict(), f"./results/sprout_model_epoch_{epoch+1}.pth")
        print(f"Model saved: ./results/sprout_model_epoch_{epoch+1}.pth")

# Save final model
torch.save(model.state_dict(), "./results/sprout_model_final.pth")
print("\n" + "=" * 50)
print("Training Complete! Final model saved to ./results/sprout_model_final.pth")

Starting SPROUT Training...

Epoch 1/10
------------------------------


Training: 100%|██████████| 100/100 [35:48<00:00, 21.48s/it]


Epoch 1 - Accuracy: 0.3325, Loss: 1.7514

Epoch 2/10
------------------------------


Training: 100%|██████████| 100/100 [51:08<00:00, 30.69s/it]


Epoch 2 - Accuracy: 0.3435, Loss: 1.6566

Epoch 3/10
------------------------------


Training:  32%|███▏      | 32/100 [12:31<30:07, 26.58s/it]

## Cell 11: Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, NUM_EPOCHS + 1), train_accuracies, 'b-o', label='Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, NUM_EPOCHS + 1), train_losses, 'r-o', label='Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('./results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved to ./results/training_curves.png")

## Cell 12: Test the Model

In [ ]:
# Load the trained model
model.load_state_dict(torch.load("./results/sprout_model_final.pth", map_location=device))
model.eval()

print("Testing the trained model...")
print("=" * 50)

# Run test episodes
test_accuracies = []
num_test_episodes = 50

for episode in tqdm(range(num_test_episodes), desc="Testing"):
    support_images, support_labels, query_images, query_labels = create_episode(
        test_dataset, N_WAY, K_SHOT, N_QUERY
    )
    
    with torch.no_grad():
        logits, prototypes = model(query_images, support_images, support_labels)
    
    _, predicted = torch.max(logits.data, 1)
    accuracy = (predicted == query_labels).float().mean().item()
    test_accuracies.append(accuracy)

print(f"\nTest Accuracy: {np.mean(test_accuracies):.4f} (+/- {np.std(test_accuracies):.4f})")
print("=" * 50)

## Done!

Your SPROUT model is trained. The results are saved in:
- `./results/sprout_model_final.pth` - Trained model weights
- `./results/training_curves.png` - Training plots